<a href="https://colab.research.google.com/github/labphonlab/phonetics-research-statistics-support/blob/main/notebooks/ch14_classification_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

このバッジから開くと、設定なしでそのまま実行できます。
書き換えを残したいときは **ファイル → ドライブにコピーを保存** を一度行ってください。


# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第14回：分類とクラスタリング（判別分析・混合分布・二峰性）


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本ファイルの公開用URLを、みだりに他の人へ共有しないこと。

本ファイルの公開用URLにアクセスすると、お手元のGoogleアカウントに本ファイルのコピーが作成される。
そのコピーはあなただけのファイルとして生成されるため、自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


実データ: Peterson & Barney (1952)（phonToolsパッケージ収録）／参照: Hartigan & Hartigan (1985) dip検定／Scrucca et al. mclust。

In [ ]:
install.packages(c("MASS", "mclust", "diptest", "phonTools"), repos = "https://cloud.r-project.org")
library(MASS)      # 線形判別分析 lda()
library(mclust)    # 正規混合分布モデル
library(diptest)   # Hartiganのdip検定（二峰性の検定）
library(phonTools) # Peterson & Barney (1952) の実測データ
library(tidyverse)


■ 「どれくらい違うか」ではなく「区別できるか」を測る：

第3回から第13回までは、条件によって平均がどれだけ動くかを推定してきた。しかし音声研究の問いには、別の形のものもある。**「この2つの音響特徴だけで、母音カテゴリを言い当てられるか」****「この分布は1つの山か、それとも2つの集団が混ざっているか」**といった問いである。前者は分類、後者はクラスタリングの問題であり、回帰とは異なる道具を使う。

この章では実測データ——Peterson & Barney (1952) の母音フォルマント——を使い、分類がどれくらいうまくいくか、そして第2回で学んだ話者正規化が分類精度にどう効くかを確かめる。

■ 線形判別分析：フォルマントから母音を言い当てる：

In [ ]:
data(pb52)

pb <- pb52 %>%
  mutate(talker_type = recode(as.character(type), m = "男性", w = "女性", c = "子ども")) %>%
  group_by(speaker) %>%
  # 第2回と同じLobanov正規化（話者ごとのzスコア）。
  # ★この時点ではまだ訓練・テストに分けていないので、正規化のパラメータ
  #   （各話者の平均・SD）は「その話者の全トークン」から計算される。
  #   Lobanov正規化はラベル（母音カテゴリ）を一切使わない教師なしの変換
  #   なので、これ自体はラベル漏洩ではない。ただし、次に行う分割の仕方に
  #   よっては、"同じ話者の別トークン"が学習に混ざるという別の問題が残る
  #   （下のコメントを参照）。
  mutate(f1z = as.numeric(scale(f1)), f2z = as.numeric(scale(f2))) %>%
  ungroup()

# ★★訓練・テストは「トークン単位」ではなく「話者単位」で分ける。
#   トークン単位でランダムに分けると（例: sample(nrow(pb), ...)）、
#   同じ話者の他のトークンが訓練データに混ざり込む。その場合、モデルは
#   「未知の話者の母音を当てる」課題ではなく「既知の話者の、まだ見ていない
#   トークンを当てる」課題を解くことになり、精度は本来より高く出る
#   （この問題を含んでいた本章の旧バージョンは修正済み）。
#   話者単位で分ければ、テスト話者はモデルにとって完全に未知の話者になる。
set.seed(5)
speakers <- unique(pb$speaker)
train_speakers <- sample(speakers, floor(0.7 * length(speakers)))
train <- pb %>% filter(speaker %in% train_speakers)
test  <- pb %>% filter(!speaker %in% train_speakers)

accuracy <- function(model, data) mean(predict(model, data)$class == data$vowel)

lda_raw    <- lda(vowel ~ f1 + f2, data = train)         # 生のHz
lda_norm   <- lda(vowel ~ f1z + f2z, data = train)       # 話者内で正規化
lda_norm_3 <- lda(vowel ~ f1z + f2z + f3, data = train)  # F3を足す（未正規化）

cat("10母音の判別・未知話者への汎化（当てずっぽうなら10%）\n")
cat("  生のHz F1,F2       :", round(accuracy(lda_raw, test), 3), "\n")
cat("  話者内正規化 F1,F2 :", round(accuracy(lda_norm, test), 3), "\n")
cat("  正規化F1,F2 + 生F3 :", round(accuracy(lda_norm_3, test), 3), "\n")

# どの母音どうしが混同されやすいか
pred <- predict(lda_norm, test)$class
confusion <- as.data.frame(table(true = test$vowel, pred = pred)) %>%
  filter(true != pred) %>% arrange(desc(Freq)) %>% head(5)
cat("\n最も混同された組み合わせ:\n")
print(confusion)

# --- 実行結果 ---
# ⚠️ 本書の初版はトークン単位の分割（同一話者が訓練・テスト両方に
#    混入しうる分割）で実行した数値（生68.4%→正規化後90.8%）を掲載して
#    いたが、査読で指摘を受けて話者単位の分割に修正した。話者単位に
#    直すと未知話者への汎化はより厳しい課題になるため、正解率は
#    上の数値より下がるはずである。この節は、出版前に上のコードを
#    実際に実行し、得られた数値でこの############部分を置き換えてから
#    確定させること（本書の方針上、実行していない数値は掲載しない）。
############################################################
#
# 【出力の読み方】
# ・話者内正規化は、生のHzより判別精度を上げるはずである。使っている
#   情報はどちらもF1とF2の2つだけで、変えたのは
#   「話者ごとの声道長の違いを取り除いたかどうか」だけである。
#   第2回で見た「正規化は情報を捨てる操作ではなく、話者差というノイズを
#   取り除く操作である」という主張が、分類精度という別の角度から裏づけられる。
# ・F3を（正規化せずに）追加したときに精度がどう動くかも確認せよ。
#   もし下がるなら、F3だけ正規化していないために、いったん取り除いた
#   話者差を持ち込み直してしまっている可能性が高い。
#   特徴量を追加するときは、**他の特徴量と同じ前処理を通す**こと。
# ・混同表が音声学的に筋の通った形をしているか確認する。
#   隣り合う母音どうしの混同（例: u と U）は自然だが、
#   もし i と A のように空間の端どうしが混同されていたら、
#   それは母音の性質ではなくコードの誤りを疑う場面である。
# ・分類精度は「音響情報だけでどこまで説明できるか」の上限を示す指標である。
#   100%に届かない分は、F1・F2だけでは説明できない部分を意味する。
#   Peterson & Barney自身、聴き手の同定率が音響的な分離より高いことを報告しており、
#   人間は持続時間や文脈など他の手がかりも使っていると考えられる。
# ・★訓練データとテストデータを話者単位で分けている理由をもう一度確認する。
#   同じ話者のトークンが訓練・テスト両方に混ざると、モデルは
#   「未知の話者の母音を当てる」のではなく「既知の話者の残りのトークンを
#   当てる」課題を解いてしまい、精度が実力以上に出る。**論文に載せる
#   精度は、実際に一般化したい対象（未知の話者か、既知の話者の新しい
#   発話か）に対応する分割で測る。** より厳密には話者単位の交差検証を使う。


■ 分布は1つの山か、2つの集団の混合か：

音声研究では「このデータは2つの下位集団に分かれているのではないか」という問いがしばしば立つ。有声破裂音のVOTが、前voicing（負のVOT）を使う発話と使わない発話に分かれる場合などである。分布が二峰かどうかは、ヒストグラムの見た目では判断できない。ビン幅を変えれば山は増えたり消えたりするからである。

In [ ]:
set.seed(9)
unimodal <- rnorm(300, mean = 50, sd = 10)                        # 1つの山
bimodal  <- c(rnorm(150, mean = 20, sd = 5), rnorm(150, 80, 8))   # 2つの山

# Hartiganのdip検定: 帰無仮説は「単峰である」
cat("単峰データ: dip検定 p =", round(dip.test(unimodal)$p.value, 4), "\n")
cat("二峰データ: dip検定 p =", round(dip.test(bimodal)$p.value, 4), "\n")

# 正規混合分布モデル: 成分がいくつあるかをBICで選ばせる
mc <- Mclust(bimodal, G = 1:3, verbose = FALSE)
cat("\nmclustが選んだ成分数:", mc$G, "\n")
cat("各成分の平均:", round(mc$parameters$mean, 2), "\n")
cat("各成分の割合:", round(mc$parameters$pro, 3), "\n")

# --- 実行結果 ---
# 単峰データ: dip検定 p = 0.9184
# 二峰データ: dip検定 p = 0
#
# mclustが選んだ成分数: 2
# 各成分の平均: 20.38 80.4
# 各成分の割合: 0.5 0.5
#
# 【出力の読み方】
# ・dip検定の帰無仮説は「分布は単峰である」なので、**p値が小さいほど
#   二峰（あるいはそれ以上）の証拠が強い**。単峰データではp = 0.9184で
#   単峰性を否定できず、二峰データではp = 0で明確に否定される。
#   期待どおりの動作である。
# ・★p値の向きを取り違えないこと。「p > 0.05 だから単峰である」とは言えない。
#   検出力が足りないだけかもしれない。言えるのは
#   「二峰だという証拠は得られなかった」までである。
# ・混合分布モデルは、より多くの情報を返す。成分数を2と判定し、
#   平均を20.38と80.4（生成時の20と80）、混合比を0.5対0.5
#   （生成時も0.5対0.5）と、いずれも正確に回収している。
# ・**dip検定と混合分布モデルは役割が違う。** dip検定は「単峰か否か」だけを
#   答える検定であり、混合分布モデルは「いくつの集団が、どこに、どの割合で
#   あるか」を推定する。まずdip検定で二峰性の証拠を確認し、
#   次に混合分布モデルで中身を記述する、という順序が使いやすい。
# ・★混合分布モデルの重大な注意点は、**成分数を増やせばどんな分布にも
#   当てはまってしまう**ことである。正規分布1つのデータに3成分を当てても、
#   当てはまり自体は良くなる。だからBICのような、複雑さに罰則を課す
#   基準で成分数を選ぶ。上では G = 1:3 の範囲でBICが最良のものを選ばせている。
# ・★さらに根本的な注意として、**統計的に2成分が選ばれたことは、
#   その2つが言語的に意味のあるカテゴリであることを意味しない。**
#   録音セッションの違い、マイクの違い、話者の下位集団など、
#   別の理由で分布が分かれることは多い。成分が何に対応するかは、
#   モデルではなくデータの素性を調べて確かめる必要がある。


■ 演習：

1. `pb52` で、話者タイプ（男性・女性・子ども）を F1・F2 から判別できるか試せ。正規化前と後で精度がどう変わるかを予想してから実行せよ。
2. 母音を /i/ と /I/ の2つだけに絞って判別し、10母音のときより精度が上がることを確認せよ。
3. `pb52` の /u/ のF2について dip検定を行い、二峰性があるか調べよ。

■ 演習の解答：

In [ ]:
# 1. 話者タイプの判別。★正規化は「話者差を消す」操作なので、
#    話者タイプを当てる課題では精度が **下がる** はずである。
#    train/testは前のセルで話者単位に分割済みのものをそのまま使う。
lda_type_raw  <- lda(talker_type ~ f1 + f2, data = train)
lda_type_norm <- lda(talker_type ~ f1z + f2z, data = train)
acc_type <- function(m, d) mean(predict(m, d)$class == d$talker_type)
cat("話者タイプの判別・未知話者への汎化（3クラス、最頻クラスは約43%）\n")
cat("  生のHz      :", round(acc_type(lda_type_raw, test), 3), "\n")
cat("  正規化後    :", round(acc_type(lda_type_norm, test), 3), "\n")

# 2. 2母音に絞る。ここも話者単位で分割する（前のセルと同じ理由）。
pb_2 <- pb %>% filter(vowel %in% c("i", "I")) %>% mutate(vowel = droplevels(vowel))
set.seed(6)
speakers_2 <- unique(pb_2$speaker)
train_speakers_2 <- sample(speakers_2, floor(0.7 * length(speakers_2)))
pb_2_train <- pb_2 %>% filter(speaker %in% train_speakers_2)
pb_2_test  <- pb_2 %>% filter(!speaker %in% train_speakers_2)
m2 <- lda(vowel ~ f1z + f2z, data = pb_2_train)
cat("\n/i/ vs /I/ の判別（2クラス、当てずっぽうは50%）:",
    round(mean(predict(m2, pb_2_test)$class == pb_2_test$vowel), 3), "\n")

# 3. /u/ のF2に二峰性があるか。dip検定はラベル分割と無関係な問い
#    （データ全体の分布の形を見るだけ）なので、訓練・テスト分割は不要。
u_f2 <- pb$f2[pb$vowel == "u"]
cat("\n/u/ のF2: n =", length(u_f2), " dip検定 p =",
    round(dip.test(u_f2)$p.value, 4), "\n")
u_f2z <- pb$f2z[pb$vowel == "u"]
cat("/u/ の正規化F2:        dip検定 p =", round(dip.test(u_f2z)$p.value, 4), "\n")

# --- 実行結果 ---
# ⚠️ 前のセルと同じ理由で、話者単位の分割に修正済みのこのコードは
#    未再実行である。出版前に実行し、以下の############部分を
#    実際の出力で置き換えること。3問目（dip検定）は分割と無関係なので、
#    旧版の数値（n = 152, 生 p = 0.8386, 正規化後 p = 0.9241）は
#    そのまま有効なはずだが、念のため再実行して確認すること。
############################################################
#
# 【出力の読み方】
# ・1問目は**予想が当たったかを確認する問題**である。正規化すると
#   話者タイプの判別精度は下がるはずである。Lobanov正規化は話者ごとの
#   平均とSDを揃える操作、つまり**話者の情報を意図的に消す操作**だから
#   である。消した情報を当てさせようとすれば精度が落ちる。
# ・ここから重要な原則が出てくる。**前処理は「良い前処理」「悪い前処理」に
#   分かれるのではなく、問いによって適否が変わる。** 母音を当てたいなら
#   正規化すべきで、話者の属性を当てたいなら正規化してはいけない。
#   同じデータ、同じ操作でも、問いが変われば答えが逆になる。
#   ★話者認識や法音声学では、まさにこの理由で正規化を行わない。
# ・話者タイプの判別精度は、必ずチャンスレベル（最頻クラス、約43%）と
#   比べる。**分類精度は必ずチャンスレベルと比べる。**
#   3クラスだから33%が基準、ではない。クラスの大きさが不均等なら、
#   最頻クラスの割合が実質的な下限になる。
# ・2問目: /i/ と /I/ のような2クラスへの絞り込みは、10母音すべてを
#   相手にしたときより判別が易しくなるはずである。
#   **分類の難しさはクラス数と、クラスどうしの距離で決まる。**
#   ★逆に言えば、**「正解率が高い」という報告は、何クラスの課題かを
#   書かなければ意味を持たない。**
# ・3問目: /u/ のF2は生でも正規化後でも p が0.84、0.92 と大きく、
#   **二峰性の証拠は無い**。/u/ は後舌母音で、英語では前寄りに発音する話者と
#   そうでない話者がいる（u-fronting）ため二峰になる可能性を考えたが、
#   1952年のこのデータでは分かれていない。
# ・★この「見つからなかった」という結果も、きちんとした答えである。
#   dip検定のp値が大きいことは「単峰だと証明された」ことではなく、
#   「このデータからは二峰の証拠が得られなかった」ことを意味する。
#   152トークンという規模で検出できる程度の二峰性は無かった、と読む。
